# 03 — Plume extraction: AQ-2016-10-28

Owner: Abd · Workstream B (Remote Sensing) · `tasks/abd.md` Section 4

Pipeline order (concept doc Section 10.5): baseline composite -> water mask -> spectral features -> anomaly detection -> remove glint/cloud/land-edge artifacts -> plume-probability raster -> manual review.

**Bottom line up front** (full reasoning in `docs/event_audit.md` Section 1a/3): the only cloud-free post-event scene (2016-11-02, +5 days) shows **no visible plume**, corroborated by an independent Landsat 8 pass and by the Kalman et al. (2025) mooring record showing the turbidity signal had already dispersed ~2.5-3.5 days before either satellite pass. The spectral anomaly computed below is real output, but it is a **coastline-hugging atmospheric/mixed-pixel artifact, not a validated plume detection** — kept here for the methodology record, not as a positive result.

Pixel access: Microsoft Planetary Computer (public, no credentials) — see `docs/event_audit.md` Section 0 and Section 1a for why (AWS Earth Search has no coverage for this tile before 2017; Copernicus Data Space needs a login for pixel bytes).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_erosion

REPO_ROOT = Path.cwd().parent
sys.path.append(str(REPO_ROOT / "scripts"))
sys.path.append(str(REPO_ROOT / "backend" / "src"))

from config import ANALYSIS_BBOX  # noqa: E402
from models import plume_segmentation as ps  # noqa: E402

%matplotlib inline

## 1. Baseline: search clear pre-event Sentinel-2 scenes

2016 was S2A-only (S2B launched April 2017), so the ~5-day revisit we're used to today was a ~10-day revisit back then. The search window here is deliberately wider than the post-event ±10-day gate window — the baseline just needs clear pre-event scenes, however far back, per `docs/event_audit.md` Section 1.

In [ ]:
baseline_items = ps.search_scenes(ANALYSIS_BBOX, "2016-06-01/2016-10-18", max_cloud=5.0)[:8]
for it in baseline_items:
    print(it.id, f"cloud={it.properties.get('eo:cloud_cover'):.2f}%")

In [ ]:
baseline_bands = []
transform = crs = None
for it in baseline_items:
    bands, transform, crs = ps.load_scene_bands(it, ANALYSIS_BBOX)
    baseline_bands.append(bands)
print("loaded", len(baseline_bands), "scenes, grid shape", baseline_bands[0]["SCL"].shape)

## 2. Water mask — SCL majority vote, not any single scene

`tasks/abd.md` assigns Abd the water mask, decoupled from Pulga's independent bathymetric mask: "derive your own from Sentinel-2's SCL band." Using a majority vote across the 8 baseline scenes (rather than one scene's SCL) avoids the mask itself shifting on the event day if a plume changes how a pixel gets classified.

In [ ]:
water_mask = ps.stable_water_mask(baseline_bands, min_fraction=0.5)
water_mask_eroded = binary_erosion(water_mask, iterations=1)
print(f"water pixels: {water_mask.sum()} / {water_mask.size} ({100*water_mask.mean():.1f}%)")

plt.figure(figsize=(5,7))
plt.imshow(water_mask, cmap="Blues")
plt.title("Stable water mask (SCL majority vote, 8 baseline scenes)")
plt.axis("off")

## 3. Baseline composite + post-event scene

In [ ]:
composite = ps.baseline_composite(baseline_bands)
baseline_idx = ps.spectral_indices(composite)

post_items = ps.search_scenes(ANALYSIS_BBOX, "2016-11-02/2016-11-03")
post_item = post_items[0]
print(post_item.id, f"cloud={post_item.properties.get('eo:cloud_cover'):.2f}%")
post_bands, post_transform, post_crs = ps.load_scene_bands(post_item, ANALYSIS_BBOX)
post_idx = ps.spectral_indices(post_bands)

## 4. AOI-water cloud % — the actual gate metric

`tasks/abd.md` is explicit that scene-level cloud-cover metadata is not the right number to gate on (it includes land). This computes it properly: cloud/shadow fraction within the water mask only, from the post-event scene's own SCL.

In [ ]:
unusable = ps.unusable_mask(post_bands)
glint = ps.glint_mask(post_bands)

aoi_water_cloud_pct = 100 * (water_mask & unusable).sum() / max(water_mask.sum(), 1)
aoi_water_glint_pct = 100 * (water_mask & glint).sum() / max(water_mask.sum(), 1)
print(f"AOI-water cloud/shadow %: {aoi_water_cloud_pct:.2f}%")
print(f"AOI-water suspected-glint %: {aoi_water_glint_pct:.2f}%")

valid_mask = water_mask_eroded & (~unusable) & (~glint)
print(f"valid pixels for anomaly: {valid_mask.sum()}")

## 5. Spectral anomaly — four candidate indices

NDSSI, NSMI, red/green ratio, and a plain red-band reflectance anomaly, all computed as (post-event index) minus (baseline composite index), restricted to `valid_mask`.

In [ ]:
anomalies = ps.anomaly(baseline_idx, post_idx, valid_mask)

fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for ax, key in zip(axes.flat, ["ndssi", "nsmi", "red_green_ratio", "red_reflectance"]):
    a = anomalies[key]
    vmax = np.nanpercentile(np.abs(a), 98)
    im = ax.imshow(a, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"Anomaly: {key}")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("AQ-2016-10-28: post(11-02) minus baseline composite, all 4 indices")

## 6. Manual QC — reading the anomaly honestly

The anomaly above is large and affects nearly the entire water body, not a localized area near the Kinnet outlet. Plotting where the top percentile actually falls (see `docs/event_audit.md` Section 1a, figure `plume_index_comparison_AQ-2016-10-28.png` in `docs/qa_screenshots/`) shows a ring hugging **both** shores of the gulf, including stretches tens of kilometres from any wadi outlet — not the localized bulge a real plume would produce.

Repeating this with a same-season single-scene baseline (2016-10-13, only 20 days before the event) and a much larger coastal erosion buffer (80 m) did **not** remove the effect — which rules out "seasonal mismatch" as the explanation. This is a mixed-pixel / atmospheric-correction artifact inherent to differencing Sentinel-2 L2A reflectance over open water across dates (Sen2Cor is land-optimized; residual aerosol and sun-angle differences between acquisitions swamp subtle water-leaving-radiance signals at basin scale) — a real limitation of this simple approach, not a bug in the water mask or index math.

**Conclusion: no real plume signal in this scene.** This agrees with the true-color inspection (both Sentinel-2 2016-11-02 and Landsat 8 2016-11-01 show nothing) and with the in-situ mooring timing (turbidity had returned to background by ~17:15 Oct 29, 2.5-3.5 days before either satellite pass).

In [ ]:
# Primary index for the probability raster: plain red-band reflectance anomaly
# (simplest, most directly physically interpretable of the four).
probability = ps.anomaly_to_probability(anomalies["red_reflectance"])
geoms, out_crs = ps.probability_to_polygons(probability, transform, crs, threshold=0.7)
print(f"{len(geoms)} polygon(s) above threshold 0.7 -- these are the coastal artifact, not a plume (see markdown above)")

## 7. What this means for the project

See `docs/event_audit.md` Section 3 for the full writeup. Short version: **NO-GO on image-based plume validation for AQ-2016-10-28**, for a real physical reason (the plume dispersed faster than the satellite revisit gap), not a data-quality problem. Recommended pivot: use the Kalman et al. (2025) in-situ mooring salinity/turbidity time series as Nizar's validation target instead of a satellite-derived mask — it's a quantitative, continuous, satellite-independent measurement of the same thing this pipeline was trying to detect qualitatively, and it already exists.